In [1]:
import pandas as pd
import numpy as np

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder

In [16]:
df=pd.read_csv("covid_toy.csv")
df.sample(4)

,age,gender,fever,cough,city,has_covid
34,74,Male,102.0,Mild,Mumbai,Yes
58,23,Male,98.0,Strong,Mumbai,Yes
54,60,Female,99.0,Mild,Mumbai,Yes
20,12,Male,98.0,Strong,Bangalore,No


In [4]:
df["cough"].unique()

array(['Mild', 'Strong'], dtype=object)

## 1.ap sab columns par indivisual tranformation alaoge. jo ki ak katin kam hai

In [5]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(df.iloc[:,:5],df.iloc[:,-1],test_size=0.2,random_state=0)

In [6]:
X_train_cough=X_train[["cough"]]
X_test_cough=X_test[["cough"]]
X_train_gender_city=X_train[["gender","city"]]
X_test_gender_city=X_test[["gender","city"]]

In [7]:
oe=OrdinalEncoder(categories=[["Mild","Strong"]])
ohe=OneHotEncoder(drop="first")

In [8]:
oe.fit(X_test_cough)
X_train_cough=oe.transform(X_train_cough)
X_test_cough=oe.transform(X_test_cough)

In [9]:
ohe.fit(X_test_gender_city[["gender","city"]])
X_train_gender_city=ohe.transform(X_train_gender_city[["gender","city"]]).toarray()
X_test_gender_city=ohe.transform(X_test_gender_city[["gender","city"]]).toarray()

In [11]:
X_train=np.hstack((X_train_cough,X_train_gender_city))
X_test=np.hstack((X_test_cough,X_test_gender_city))


In [13]:
columns=["cough"]+list(ohe.get_feature_names_out(["gender","city"]))
columns

['cough', 'gender_Male', 'city_Delhi', 'city_Kolkata', 'city_Mumbai']

In [14]:
X_train=pd.DataFrame(X_train,columns=columns)
X_test=pd.DataFrame(X_test,columns=columns)


In [15]:
X_train

,cough,gender_Male,city_Delhi,city_Kolkata,city_Mumbai
0,0.0,0.0,0.0,0.0,0.0
1,1.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,1.0,0.0
3,1.0,0.0,1.0,0.0,0.0
4,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...
75,1.0,0.0,0.0,1.0,0.0
76,0.0,1.0,0.0,0.0,0.0
77,0.0,1.0,0.0,0.0,1.0
78,0.0,0.0,0.0,0.0,0.0


## 2. ye transformer ki power 

In [17]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(df.iloc[:,:5],df.iloc[:,-1],test_size=0.2,random_state=0)

In [18]:
from sklearn.compose import ColumnTransformer


In [24]:
transformer=ColumnTransformer(transformers=[
    ("tnf1",SimpleImputer(),['fever']),
    ("tnf2",OrdinalEncoder(categories=[['Mild','Strong']]),["cough"]),
    ("tnf3",OneHotEncoder(drop="first",sparse_output=False),["gender",'city'])
],remainder="passthrough")

In [25]:
transformer.fit_transform(X_train)

array([[ 99.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  22.        ],
       [104.        ,   1.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  56.        ],
       [ 98.        ,   0.        ,   0.        ,   0.        ,
          1.        ,   0.        ,  31.        ],
       [104.        ,   1.        ,   0.        ,   1.        ,
          0.        ,   0.        ,  75.        ],
       [ 99.        ,   0.        ,   1.        ,   0.        ,
          0.        ,   0.        ,  72.        ],
       [ 99.        ,   1.        ,   1.        ,   0.        ,
          0.        ,   0.        ,  66.        ],
       [101.        ,   1.        ,   1.        ,   0.        ,
          0.        ,   0.        ,  14.        ],
       [ 98.        ,   1.        ,   0.        ,   0.        ,
          1.        ,   0.        ,  10.        ],
       [ 98.        ,   0.        ,   1.        ,   0.        ,
          1.    